## Example 1: Basic cached_property Decorator

In [ ]:
from cached_property import cached_property
import time

print("Example 1: Basic cached_property Decorator")

class ExpensiveCalculation:
    def __init__(self, name):
        self.name = name
    
    @cached_property
    def expensive_value(self):
        print(f"Computing expensive value for {self.name}...")
        time.sleep(0.1)  # Simulate expensive computation
        return f"Result for {self.name}: {hash(self.name) % 1000}"

try:
    # Create instance
    calc = ExpensiveCalculation("test")
    print(f"✓ Created instance: {calc.name}")
    
    # First access - will compute
    print("First access:")
    result1 = calc.expensive_value
    print(f"✓ Result: {result1}")
    
    # Second access - will use cache
    print("Second access (should be cached):")
    result2 = calc.expensive_value
    print(f"✓ Result: {result2}")
    
    # Verify results are identical
    assert result1 == result2, (result1, result2)
    print("✓ Results match")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise AssertionError(
        f"cached_property example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Threaded cached_property

In [ ]:
from cached_property import threaded_cached_property
import threading
import time

print("Example 2: Threaded cached_property")

class ThreadSafeCalculation:
    def __init__(self, name):
        self.name = name
    
    @threaded_cached_property
    def thread_safe_value(self):
        print(f"Computing thread-safe value for {self.name}...")
        time.sleep(0.05)
        return f"Thread-safe result: {threading.get_ident()}"

try:
    calc = ThreadSafeCalculation("thread-test")
    print(f"✓ Created thread-safe instance: {calc.name}")
    
    # Access from main thread
    result1 = calc.thread_safe_value
    print(f"✓ First access: {result1}")
    
    # Access again - should be cached
    result2 = calc.thread_safe_value
    print(f"✓ Second access: {result2}")
    
    assert result1 == result2, (result1, result2)
    print("✓ Thread-safe caching works")
    print(f"✓ Threaded cached_property ensures thread safety")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise AssertionError(
        f"cached_property example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: TTL (Time-to-Live) cached_property

In [ ]:
from cached_property import cached_property_with_ttl
import time

print("Example 3: TTL (Time-to-Live) cached_property")

class TimedCache:
    def __init__(self, name):
        self.name = name
    
    @cached_property_with_ttl(ttl=1)  # Cache for 1 second
    def timed_value(self):
        current_time = time.time()
        print(f"Computing timed value at {current_time:.2f}")
        return f"Timestamp: {current_time:.2f}"

try:
    cache = TimedCache("timed-test")
    print(f"✓ Created timed cache instance: {cache.name}")
    
    # First access
    result1 = cache.timed_value
    print(f"✓ First result: {result1}")
    
    # Immediate second access - should be cached
    result2 = cache.timed_value
    print(f"✓ Immediate second result: {result2}")
    assert result1 == result2, (result1, result2)
    print("✓ Values match (cached)")
    
    # Wait and access again - should recompute due to TTL expiration
    print("Waiting 1.2 seconds for TTL expiration...")
    time.sleep(1.2)
    result3 = cache.timed_value
    print(f"✓ After TTL expiration: {result3}")
    assert result1 != result3, (result1, result3)
    print("✓ Value changed (TTL expired)")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise AssertionError(
        f"cached_property example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Cache Invalidation and Management

In [ ]:
from cached_property import cached_property
import time

print("Example 4: Cache Invalidation and Management")

class ManagedCache:
    def __init__(self, name):
        self.name = name
        self.counter = 0
    
    @cached_property
    def managed_value(self):
        self.counter += 1
        print(f"Computing managed value (call #{self.counter})")
        return f"Value-{self.counter}: {time.time():.2f}"
    
    def invalidate_cache(self):
        """Manually clear the cached property.

        cached_property stores the computed value in the instance __dict__ under the
        property's own name, so removing that entry makes the next access recompute it.
        """
        self.__dict__.pop('managed_value', None)
        print("✓ Cache invalidated")

try:
    cache = ManagedCache("managed-test")
    print(f"✓ Created managed cache: {cache.name}")
    
    # First access
    result1 = cache.managed_value
    print(f"✓ First access: {result1}")
    
    # Second access - cached
    result2 = cache.managed_value
    print(f"✓ Second access (cached): {result2}")
    assert result1 == result2 and cache.counter == 1, (result1, result2, cache.counter)
    
    # Invalidate cache manually
    cache.invalidate_cache()
    
    # Third access - will recompute
    result3 = cache.managed_value
    print(f"✓ After invalidation: {result3}")
    
    assert result1 != result3 and cache.counter == 2, (result1, result3, cache.counter)
    print("✓ Cache invalidation works")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise AssertionError(
        f"cached_property example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Multiple cached_property Variants

In [ ]:
from cached_property import (
    cached_property, 
    threaded_cached_property,
    cached_property_with_ttl
)

print("Example 5: Multiple cached_property Variants")

class MultiCache:
    def __init__(self, name):
        self.name = name
    
    @cached_property
    def simple_cache(self):
        return f"Simple cached value for {self.name}"
    
    @threaded_cached_property
    def thread_safe_cache(self):
        return f"Thread-safe cached value for {self.name}"
    
    @cached_property_with_ttl(ttl=5)
    def ttl_cache(self):
        return f"TTL cached value for {self.name} at {time.time():.2f}"

try:
    import time
    
    cache = MultiCache("multi-test")
    print(f"✓ Created multi-cache instance: {cache.name}")
    
    # Test all cache types
    simple = cache.simple_cache
    thread_safe = cache.thread_safe_cache
    ttl = cache.ttl_cache
    
    print(f"✓ Simple cache: {simple}")
    print(f"✓ Thread-safe cache: {thread_safe}")
    print(f"✓ TTL cache: {ttl}")
    
    # Verify they're all cached on second access
    assert cache.simple_cache == simple, (cache.simple_cache, simple)
    assert cache.thread_safe_cache == thread_safe, (cache.thread_safe_cache, thread_safe)
    assert cache.ttl_cache == ttl, (cache.ttl_cache, ttl)
    print("✓ Simple, thread-safe and TTL values are all served from the cache")
    
    # Show available decorators
    decorators = ['cached_property', 'threaded_cached_property', 'cached_property_with_ttl']
    print(f"✓ Available decorators: {decorators}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise AssertionError(
        f"cached_property example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e